# Module 1 — Ingestion: PDF → Neo4j Document Graph

**What we build in this module:**
- Parse 10-K filings with **Docling** (threaded PDF pipeline, table structure, OCR)
- Split each filing into token-aware **chunks** with `HybridChunker`
- Store the result in Neo4j as a `(Company)-[:HAS_DOCUMENT]->(Document)-[:HAS_CHUNK]->(Chunk)` graph
- Apply schema constraints and a vector index ready for embeddings in Module 2

**Services introduced:**
- `services.docling_service` — PDF conversion + chunking singleton
- `services.neo4j_service` — Neo4j driver singleton
- `ingestion.docling_loader.load_filing` — PDF → `list[Document]`
- `ingestion.graph_writer.write_documents` — `list[Document]` → Neo4j
- `ingestion.schema.apply_schema` — constraints + indexes

## 1. Setup

In [1]:
from pathlib import Path

from financial_advisor.ingestion.docling_loader import load_filing
from financial_advisor.ingestion.graph_writer import write_documents
from financial_advisor.ingestion.schema import apply_schema
from financial_advisor.services.docling_service import docling_service
from financial_advisor.services.neo4j_service import neo4j_service

FILINGS_DIR = Path("../data/filings")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

In [2]:
# Verify Neo4j connectivity before we go further
ok = neo4j_service.check_connection()
print("Neo4j connected:", ok)
assert ok, "Cannot reach Neo4j — check NEO4J_URI / credentials in .env"

[neo4j] connected — database: packt
Neo4j connected: True


## 2. Apply Graph Schema

Creates uniqueness constraints on `Company.id`, `Document.id`, `Chunk.id`  
and a vector index on `Chunk.embedding` (dimensions=1536, cosine similarity).

Running this multiple times is safe — all statements use `IF NOT EXISTS`.

In [3]:
apply_schema(module=1)
print("Schema applied.")

  [schema] OK  company_id
  [schema] OK  document_id
  [schema] OK  chunk_id
  [schema] OK  chunk_text
  [schema] OK  chunk_embedding
[schema] 5/5 statements applied — all good
Schema applied.


## 3. Discover Filings

Filings follow the naming convention `{COMPANY}_{YEAR}_{TYPE}.pdf`  
e.g. `APPLE_2018_10K.pdf`, `3M_2018_10K.pdf`

In [ ]:
def parse_filing_name(stem: str) -> tuple[str, int] | None:
    """Return (company_id, year) from stems like 'APPLE_2018_10K'."""
    parts = stem.split("_")
    for i, part in enumerate(parts):
        if part.isdigit() and len(part) == 4:
            return "_".join(parts[:i]) or stem, int(part)
    return None


pdfs = sorted(FILINGS_DIR.glob("**/*.pdf"))
filings = []
for pdf in pdfs:
    parsed = parse_filing_name(pdf.stem)
    if parsed:
        company_id, year = parsed
        filings.append({"path": pdf, "company_id": company_id, "year": year})
        print(f"  {pdf.name}  →  company={company_id}, year={year}")
    else:
        print(f"  [SKIP] {pdf.name} — cannot parse company/year")

print(f"\n{len(filings)} filing(s) ready to ingest.")

  3M_2018_10K.pdf  →  company=3M, year=2018
  APPLE_2018_10K.pdf  →  company=APPLE, year=2018

2 filing(s) ready to ingest.


## 4. Convert a Single Filing with Docling

Before bulk-ingesting everything, let's inspect one filing step by step  
to understand what Docling extracts and how the chunker splits the document.

In [5]:
# Pick the first filing for the walkthrough
sample = filings[0]
print(f"Processing: {sample['path'].name}")

conv_result = docling_service.convert_pdf(sample["path"])
print("Conversion status:", getattr(conv_result, "status", "ok"))

Processing: 3M_2018_10K.pdf
Conversion status: ConversionStatus.SUCCESS


In [6]:
# Document-level metadata extracted from the PDF
meta = docling_service.extract_metadata(conv_result, sample["path"])
for k, v in meta.items():
    print(f"  {k}: {v}")

  doc_name: 3M_2018_10K.pdf
  title: 3M_2018_10K
  source: /Users/ale/Dropbox/prj-graphaware/packt-course-beyond-graphrag/data/filings/3M_2018_10K.pdf
  format: pdf
  total_pages: 160
  author: None
  subject: None
  creation_date: None
  keywords: None


In [7]:
# Chunk the document and inspect the first few chunks
chunks = docling_service.chunk_document(conv_result)
print(f"Total chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} (pages {chunk.pages}) ---")
    print(chunk.text[:400])
    print()

Total chunks: 283

--- Chunk 0 (pages [1]) ---
low

--- Chunk 1 (pages [1]) ---
UNITED STATES SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549

--- Chunk 2 (pages [1]) ---
FORM 10-K
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2018
Commission file number 1-3285



## 5. Load a Single Filing into Neo4j

`load_filing` wraps conversion + chunking and returns LangChain `Document` objects  
ready to be passed to `write_documents`.

In [6]:
documents = load_filing(
    sample["path"],
    company_id=sample["company_id"],
    year=sample["year"],
)
print(f"Documents (chunks) produced: {len(documents)}")

# Inspect metadata on one document
print("\nSample metadata:")
for k, v in documents[0].metadata.items():
    print(f"  {k}: {v}")

Documents (chunks) produced: 283

Sample metadata:
  company_id: 3M
  year: 2018
  chunk_index: 0
  pages: [1]
  doc_name: 3M_2018_10K.pdf
  title: 3M_2018_10K
  source: /Users/ale/Dropbox/prj-graphaware/packt-course-beyond-graphrag/data/filings/3M_2018_10K.pdf
  format: pdf
  total_pages: 160
  author: None
  subject: None
  creation_date: None
  keywords: None


In [7]:
# Write to Neo4j
write_documents(documents, company_id=sample["company_id"])
print("Done.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (row) { ... }', position=<SummaryInputPosition line=3, column=13, offset=45>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 45, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            UNWIND $rows AS row\n            CALL {\n                WITH row\n                MATCH (d:Document {id: row.doc_id})\n                CREATE (ch:Chunk {\n                    id:         row.id,\n                    text:       row.text,\n                    idx:        row.idx,\n                    pages:      row.pages,\n

  Wrote 283 chunks for 3M_2018_10K.pdf
Done.


## 6. Verify — Query the Graph

Check that the nodes and relationships were created correctly.

In [8]:
# Node counts
counts = neo4j_service.run_query("""
    MATCH (c:Company) WITH count(c) AS companies
    MATCH (d:Document) WITH companies, count(d) AS documents
    MATCH (ch:Chunk)   RETURN companies, documents, count(ch) AS chunks
""")
print(counts[0])

{'companies': 1, 'documents': 1, 'chunks': 283}


In [9]:
# Sample the graph: one company, its documents, first 3 chunks
rows = neo4j_service.run_query("""
    MATCH (c:Company)-[:HAS_DOCUMENT]->(d:Document)-[:HAS_CHUNK]->(ch:Chunk)
    WHERE c.id = $company_id
    RETURN c.id AS company, d.doc_name AS document, d.total_pages AS pages,
           ch.idx AS chunk_idx, ch.pages AS chunk_pages,
           left(ch.text, 120) AS snippet
    ORDER BY ch.idx
    LIMIT 3
""", {"company_id": sample["company_id"]})

for row in rows:
    print(row)

{'company': '3M', 'document': '3M_2018_10K.pdf', 'pages': 160, 'chunk_idx': 0, 'chunk_pages': [1], 'snippet': 'low'}
{'company': '3M', 'document': '3M_2018_10K.pdf', 'pages': 160, 'chunk_idx': 1, 'chunk_pages': [1], 'snippet': 'UNITED STATES SECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549'}
{'company': '3M', 'document': '3M_2018_10K.pdf', 'pages': 160, 'chunk_idx': 2, 'chunk_pages': [1], 'snippet': 'FORM 10-K\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year end'}


## 7. Ingest All Filings

Now run the full loop over every PDF found in `data/filings/`.  
Already-ingested documents are skipped automatically (idempotent).

In [12]:
for filing in filings:
    print(f"\n[{filing['company_id']}] {filing['path'].name} (year={filing['year']})")
    docs = load_filing(
        filing["path"],
        company_id=filing["company_id"],
        year=filing["year"],
    )
    if not docs:
        print("  No chunks produced — skipping.")
        continue
    print(f"  {len(docs)} chunks extracted")
    write_documents(docs, company_id=filing["company_id"])

print("\nAll filings processed.")


[3M] 3M_2018_10K.pdf (year=2018)
  283 chunks extracted
  Skipping 3M_2018_10K.pdf — already in graph

[APPLE] APPLE_2018_10K.pdf (year=2018)
  No chunks produced — skipping.

All filings processed.


In [11]:
# Final count across all companies
summary = neo4j_service.run_query("""
    MATCH (c:Company)-[:HAS_DOCUMENT]->(d:Document)
    OPTIONAL MATCH (d)-[:HAS_CHUNK]->(ch:Chunk)
    RETURN c.id AS company, d.doc_name AS document,
           d.total_pages AS pages, count(ch) AS chunks
    ORDER BY company, document
""")

for row in summary:
    print(row)

{'company': '3M', 'document': '3M_2018_10K.pdf', 'pages': 160, 'chunks': 283}
